# Molecular Cluster Analysis — Scotland

Characterises molecular clusters using spatial, temporal, demographic, socioeconomic, geographic, and lineage features; classifies them into typologies; selects representative and extreme examples; and dissects chosen clusters in detail.

**Input columns:**

| Column | Description |
|---|---|
| `sequence_id` | Unique sequence identifier |
| `cluster_id` | Molecular cluster assignment |
| `collection_date` | Sample collection date |
| `wn_mid_date` | Window midpoint date (constant within cluster/window) |
| `sex` | Biological sex |
| `age_band` | Age band |
| `datazone` | Scottish datazone identifier |
| `dz_xcoord` | Datazone centroid X (British National Grid, metres) |
| `dz_ycoord` | Datazone centroid Y (British National Grid, metres) |
| `dz_simd_quintile` | SIMD deprivation quintile (1 = most deprived, 5 = least deprived) |
| `health_board` | NHS Scotland health board (14 boards) |
| `pango_lineage` | Pango viral lineage |

**Workflow:**
1. Load data
2. Clean and filter
3. Spatial summaries per cluster
4. **Temporal summaries per cluster** *(new)*
5. Demographic summaries (age, sex)
6. Socioeconomic summaries (SIMD)
7. Geographic summaries (health board)
8. Lineage summaries (Pango)
9. Combined cluster-level table + exploratory plots
10. Interpretable typology labels
11. Rule-based cluster selection
12. KMeans meta-typology
13. Cluster dissection — case-level plots
14. Enrichment heatmaps
15. Catalogue summary statistics
16. Export


---
## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin_min

from utils import data as load_data

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.3f}'.format)
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

---
## 0. Data
columns: `sequence_id`, `cluster_id`, `sex`, `age_band`, `datazone`,
> `dz_xcoord`, `dz_ycoord`, `dz_simd_quintile`, `health_board`, `pango_lineage`

In [ ]:
df_raw = load_data.load_analysis_columns(
    ["sequence_id", "cluster_id", "pango_lineage",
     "sex", "age_band", "datazone", "collection_date", "wn_mid_date",
     "dz_xcoord", "dz_ycoord", "dz_simd_quintile"
     ]
).to_pandas()

# Parse date columns
df_raw["collection_date"] = pd.to_datetime(df_raw["collection_date"])
df_raw["wn_mid_date"]     = pd.to_datetime(df_raw["wn_mid_date"])

location = pd.read_csv("../data/raw/datazone/2020v2_simd.csv")
location["datazone"]     = location["DZ"]
location["health_board"] = location["HBname"]
location = location[["datazone", "health_board"]].copy()

df_raw = df_raw.merge(location, on="datazone", how="left")


In [ ]:
df_raw.head()

---
## 1. Clean and Filter

In [ ]:
df = df_raw.copy()

df = df.dropna(subset=['cluster_id', 'sequence_id', 'dz_xcoord', 'dz_ycoord'])

df['cluster_id']       = df['cluster_id'].astype(str)
df['sex']              = df['sex'].astype('category')
df['health_board']     = df['health_board'].astype('category')
df['pango_lineage']    = df['pango_lineage'].astype('category')
df['dz_simd_quintile'] = pd.Categorical(
    df['dz_simd_quintile'], categories=[1, 2, 3, 4, 5], ordered=True
)
df['age_band'] = pd.Categorical(
    df['age_band'], categories=df_raw["age_band"].unique().tolist(), ordered=True
)

cluster_sizes = df.groupby('cluster_id').size().rename('cluster_size')
df = df.merge(cluster_sizes, on='cluster_id')

# ── Minimum cluster size for detailed analysis ──────────────────────────────
MIN_CLUSTER_SIZE = 5
# ────────────────────────────────────────────────────────────────────────────

df_analysis = df[df['cluster_size'] >= MIN_CLUSTER_SIZE].copy()

print(f'Full dataset:     {df.shape[0]:>6,} sequences | {df["cluster_id"].nunique():>5,} clusters')
print(f'Analysis subset:  {df_analysis.shape[0]:>6,} sequences | '
      f'{df_analysis["cluster_id"].nunique():>5,} clusters (size >= {MIN_CLUSTER_SIZE})')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sizes_all = df.groupby('cluster_id').size()
axes[0].hist(sizes_all, bins=40, edgecolor='white', color='steelblue')
axes[0].set_xlabel('Cluster size'); axes[0].set_ylabel('Clusters')
axes[0].set_title('Cluster size distribution')
axes[1].hist(np.log10(sizes_all + 1), bins=40, edgecolor='white', color='steelblue')
axes[1].set_xlabel('log10(cluster size + 1)'); axes[1].set_ylabel('Clusters')
axes[1].set_title('Cluster size (log scale)')
plt.tight_layout(); plt.show()
df_analysis["cluster_size"].describe().to_frame().T

---
## 2. Spatial Summaries per Cluster

Coordinates are datazone centroids on the British National Grid (metres).

In [ ]:
def cluster_spatial_summary(g):
    x = g['dz_xcoord'].to_numpy()
    y = g['dz_ycoord'].to_numpy()
    cx, cy = x.mean(), y.mean()
    dist = np.sqrt((x - cx)**2 + (y - cy)**2)
    return pd.Series({
        'n_cases':              len(g),
        'centroid_x':           cx,
        'centroid_y':           cy,
        'mean_dist_centroid':   dist.mean(),
        'median_dist_centroid': np.median(dist),
        'max_dist_centroid':    dist.max(),
        'sd_x':                 x.std(ddof=1) if len(g) > 1 else 0.0,
        'sd_y':                 y.std(ddof=1) if len(g) > 1 else 0.0,
        'n_datazones':          g['datazone'].nunique(),
    })

cluster_spatial = (
    df_analysis.groupby('cluster_id', observed=True)
    .apply(cluster_spatial_summary, include_groups=False)
    .reset_index()
)
print(f'Spatial summary: {cluster_spatial.shape}')
cluster_spatial.head()

---
## 2b. Temporal Summaries per Cluster

`collection_date` records when each sample was collected; `wn_mid_date` is the window midpoint (constant for every sequence in the same window). Duration is bounded by the three-week window width, so `date_range_days` captures intra-window spread, not epidemic duration.

In [ ]:
def cluster_temporal_summary(g):
    dates  = g['collection_date'].dropna()
    wn_mid = g['wn_mid_date'].dropna().iloc[0] if g['wn_mid_date'].notna().any() else pd.NaT
    if dates.empty:
        return pd.Series({
            'min_collection_date':    pd.NaT,
            'max_collection_date':    pd.NaT,
            'median_collection_date': pd.NaT,
            'date_range_days':        float('nan'),
            'wn_mid_date':            wn_mid,
        })
    min_d    = dates.min()
    max_d    = dates.max()
    median_d = dates.sort_values().iloc[len(dates) // 2]
    return pd.Series({
        'min_collection_date':    min_d,
        'max_collection_date':    max_d,
        'median_collection_date': median_d,
        'date_range_days':        (max_d - min_d).days,
        'wn_mid_date':            wn_mid,
    })


cluster_temporal = (
    df_analysis.groupby('cluster_id', observed=True)
    .apply(cluster_temporal_summary, include_groups=False)
    .reset_index()
)
print(f'Temporal summary: {cluster_temporal.shape}')
cluster_temporal.head()


---
## 3. Demographic Summaries (Age and Sex)

In [ ]:
def entropy_from_counts(counts):
    """Shannon entropy. 0 = perfectly homogeneous."""
    p = counts / counts.sum()
    p = p[p > 0]
    return -(p * np.log(p)).sum()


def observed_value_counts(series, dropna=True, normalize=False):
    if hasattr(series, 'cat'):
        series = series.cat.remove_unused_categories()
    return series.value_counts(dropna=dropna, normalize=normalize)


def cluster_demographic_summary(g):
    sex_counts = observed_value_counts(g['sex'], dropna=False)
    age_counts = observed_value_counts(g['age_band'], dropna=False)
    return pd.Series({
        'dominant_sex':           sex_counts.idxmax(),
        'dominant_sex_prop':      sex_counts.max() / sex_counts.sum(),
        'sex_entropy':            entropy_from_counts(sex_counts),
        'dominant_age_band':      age_counts.idxmax(),
        'dominant_age_band_prop': age_counts.max() / age_counts.sum(),
        'age_entropy':            entropy_from_counts(age_counts),
        'n_age_bands_present':    age_counts.size,
    })


cluster_demo = (
    df_analysis.groupby('cluster_id', observed=True)
    .apply(cluster_demographic_summary, include_groups=False)
    .reset_index()
)
print(f'Demographic summary: {cluster_demo.shape}')
cluster_demo.head()

---
## 4. Socioeconomic Summaries (SIMD Quintile)

SIMD quintile: 1 = most deprived, 5 = least deprived.

In [ ]:
def cluster_simd_summary(g):
    simd   = g['dz_simd_quintile'].astype(int)
    counts = simd.value_counts().reindex([1,2,3,4,5], fill_value=0)
    return pd.Series({
        'dominant_simd_quintile': counts.idxmax(),
        'dominant_simd_prop':     counts.max() / counts.sum(),
        'simd_entropy':           entropy_from_counts(counts),
        'prop_simd_q1':           counts[1] / counts.sum(),
        'prop_simd_q5':           counts[5] / counts.sum(),
        'mean_simd':              simd.mean(),
        'median_simd':            simd.median(),
        'n_simd_quintiles':       (counts > 0).sum(),
    })


cluster_simd = (
    df_analysis.groupby('cluster_id', observed=True)
    .apply(cluster_simd_summary, include_groups=False)
    .reset_index()
)
print(f'SIMD summary: {cluster_simd.shape}')
cluster_simd.head()

---
## 5. Geographic Summaries (Health Board)

In [ ]:
def cluster_hb_summary(g):
    hb_counts = observed_value_counts(g['health_board'], dropna=False)
    return pd.Series({
        'dominant_health_board': hb_counts.idxmax(),
        'dominant_hb_prop':      hb_counts.max() / hb_counts.sum(),
        'hb_entropy':            entropy_from_counts(hb_counts),
        'n_health_boards':       hb_counts.size,
        'is_multi_board':        int(hb_counts.size > 1),
    })


cluster_hb = (
    df_analysis.groupby('cluster_id', observed=True)
    .apply(cluster_hb_summary, include_groups=False)
    .reset_index()
)
print(f'Health board summary: {cluster_hb.shape}')
cluster_hb.head()

---
## 6. Lineage Summaries (Pango)

In [ ]:
def cluster_lineage_summary(g):
    lin_counts = observed_value_counts(g['pango_lineage'], dropna=False)
    return pd.Series({
        'dominant_lineage':      lin_counts.idxmax(),
        'dominant_lineage_prop': lin_counts.max() / lin_counts.sum(),
        'lineage_entropy':       entropy_from_counts(lin_counts),
        'n_lineages':            lin_counts.size,
        'is_mixed_lineage':      int(lin_counts.size > 1),
    })


cluster_lineage = (
    df_analysis.groupby('cluster_id', observed=True)
    .apply(cluster_lineage_summary, include_groups=False)
    .reset_index()
)
print(f'Lineage summary: {cluster_lineage.shape}')
cluster_lineage.head()

---
## 7. Combined Cluster-Level Summary Table

In [ ]:
cluster_summary = (
    cluster_spatial
    .merge(cluster_temporal, on='cluster_id')
    .merge(cluster_demo,     on='cluster_id')
    .merge(cluster_simd,     on='cluster_id')
    .merge(cluster_hb,       on='cluster_id')
    .merge(cluster_lineage,  on='cluster_id')
)

print(f'Cluster summary: {cluster_summary.shape[0]} clusters x {cluster_summary.shape[1]} columns')
cluster_summary.head()


### 8a. Exploratory distributions

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 12))
axes = axes.flatten()

specs = [
    ('n_cases',              'Cluster size',                False),
    ('median_dist_centroid', 'Median dist to centroid (m)', True),
    ('date_range_days',      'Date range (days)',           False),
    ('age_entropy',          'Age entropy',                 False),
    ('sex_entropy',          'Sex entropy',                 False),
    ('simd_entropy',         'SIMD entropy',                False),
    ('mean_simd',            'Mean SIMD quintile',          False),
    ('lineage_entropy',      'Lineage entropy',             False),
    ('n_health_boards',      'Health boards per cluster',   False),
]
for ax, (col, label, log_x) in zip(axes, specs):
    data = np.log10(cluster_summary[col] + 1) if log_x else cluster_summary[col]
    lbl  = f'log10({label}+1)' if log_x else label
    ax.hist(data.dropna(), bins=30, edgecolor='white', color='steelblue')
    ax.set_xlabel(lbl, fontsize=8); ax.set_ylabel('Clusters'); ax.set_title(lbl, fontsize=8)

plt.suptitle('Cluster-level feature distributions', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].scatter(cluster_summary['n_cases'], cluster_summary['median_dist_centroid'],
                alpha=0.35, s=18, color='steelblue')
axes[0].set_xlabel('Cluster size'); axes[0].set_ylabel('Median dist centroid (m)')
axes[0].set_title('Size vs spatial spread')

sc = axes[1].scatter(cluster_summary['n_cases'], cluster_summary['age_entropy'],
                     c=cluster_summary['mean_simd'], cmap='RdYlGn', alpha=0.5, s=18)
plt.colorbar(sc, ax=axes[1], label='Mean SIMD quintile\n(1=deprived, 5=affluent)')
axes[1].set_xlabel('Cluster size'); axes[1].set_ylabel('Age entropy')
axes[1].set_title('Size vs age diversity (colour=SIMD)')

sc2 = axes[2].scatter(cluster_summary['simd_entropy'], cluster_summary['lineage_entropy'],
                      c=cluster_summary['n_cases'], cmap='plasma', alpha=0.5, s=18)
plt.colorbar(sc2, ax=axes[2], label='Cluster size')
axes[2].set_xlabel('SIMD entropy'); axes[2].set_ylabel('Lineage entropy')
axes[2].set_title('Deprivation diversity vs lineage diversity')

plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4))

hb_counts = (
    cluster_summary.groupby('dominant_health_board')['cluster_id']
    .count().sort_values(ascending=True)
)
hb_counts.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Number of clusters')
ax.set_title('Clusters per dominant health board')
ax.tick_params(axis='y', labelsize=8)

plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

simd_overall = df['dz_simd_quintile'].astype(int).value_counts().sort_index()
axes[0].bar(simd_overall.index, simd_overall.values,
            color=['#d73027','#fc8d59','#fee090','#91bfdb','#4575b4'],
            edgecolor='white')
axes[0].set_xticks([1,2,3,4,5])
axes[0].set_xticklabels(['Q1\n(most\ndeprived)','Q2','Q3','Q4','Q5\n(least\ndeprived)'])
axes[0].set_ylabel('Sequences')
axes[0].set_title('SIMD quintile distribution (all sequences)')

simd_by_hb = (
    df.groupby('health_board', observed=True)['dz_simd_quintile']
    .apply(lambda x: x.astype(int).mean())
    .sort_values()
)
simd_by_hb.plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].axvline(3, color='grey', linestyle='--', linewidth=0.8, label='Neutral (Q3)')
axes[1].set_xlabel('Mean SIMD quintile')
axes[1].set_title('Mean SIMD by health board')
axes[1].legend(fontsize=8)

plt.tight_layout(); plt.show()

---
## 9. Interpretable Typology Labels

In [ ]:
size_q90       = cluster_summary['n_cases'].quantile(0.90)
size_q99       = cluster_summary['n_cases'].quantile(0.99)
compact_q25    = cluster_summary['median_dist_centroid'].quantile(0.25)
dispersed_q75  = cluster_summary['median_dist_centroid'].quantile(0.75)
age_homo_q25   = cluster_summary['age_entropy'].quantile(0.25)
age_mixed_q75  = cluster_summary['age_entropy'].quantile(0.75)
simd_dep_q25   = cluster_summary['mean_simd'].quantile(0.25)
simd_aff_q75   = cluster_summary['mean_simd'].quantile(0.75)

print('Thresholds:')
print(f'  Size         | 90th: {size_q90:.0f}  | 99th: {size_q99:.0f}')
print(f'  Compactness  | compact <= {compact_q25:.0f} m | dispersed >= {dispersed_q75:.0f} m')
print(f'  Age entropy  | homo <= {age_homo_q25:.3f} | mixed >= {age_mixed_q75:.3f}')
print(f'  Mean SIMD    | deprived <= {simd_dep_q25:.2f} | affluent >= {simd_aff_q75:.2f}')

cluster_summary['size_type'] = np.select(
    [cluster_summary['n_cases'] >= size_q99,
     cluster_summary['n_cases'] >= size_q90],
    ['very_large', 'large'], default='moderate_or_small'
)

cluster_summary['spatial_type'] = np.select(
    [cluster_summary['median_dist_centroid'] <= compact_q25,
     cluster_summary['median_dist_centroid'] >= dispersed_q75],
    ['spatially_compact', 'spatially_dispersed'], default='intermediate_spread'
)

cluster_summary['age_type'] = np.select(
    [cluster_summary['age_entropy'] <= age_homo_q25,
     cluster_summary['age_entropy'] >= age_mixed_q75],
    ['age_homogeneous', 'age_mixed'], default='intermediate_age_mix'
)

cluster_summary['deprivation_type'] = np.select(
    [cluster_summary['mean_simd'] <= simd_dep_q25,
     cluster_summary['mean_simd'] >= simd_aff_q75],
    ['predominantly_deprived', 'predominantly_affluent'], default='mixed_deprivation'
)

cluster_summary['geographic_type'] = np.where(
    cluster_summary['n_health_boards'] > 1, 'multi_board', 'single_board'
)

cluster_summary['lineage_type'] = np.where(
    cluster_summary['n_lineages'] == 1, 'lineage_pure', 'lineage_mixed'
)


# ── Temporal typology ──────────────────────────────────────────────────────
dur_q25 = cluster_summary['date_range_days'].quantile(0.25)
dur_q75 = cluster_summary['date_range_days'].quantile(0.75)
print(f'  Date range   | short <= {dur_q25:.0f} d | long >= {dur_q75:.0f} d')

cluster_summary['temporal_type'] = np.select(
    [cluster_summary['date_range_days'] <= dur_q25,
     cluster_summary['date_range_days'] >= dur_q75],
    ['short_lived', 'long_lived'], default='intermediate_duration'
)

for label, col in [
    ('Size',        'size_type'),
    ('Temporal',    'temporal_type'),
    ('Spatial',     'spatial_type'),
    ('Age',         'age_type'),
    ('Deprivation', 'deprivation_type'),
    ('Geographic',  'geographic_type'),
    ('Lineage',     'lineage_type'),
]:
    print(f'\n{label} type breakdown:')
    print(cluster_summary[col].value_counts().to_string())

---
## 10. Rule-Based Cluster Selection

In [ ]:
TOP_N = 3

shortlist = pd.concat({
    'largest': (
        cluster_summary.sort_values('n_cases', ascending=False).head(TOP_N)
    ),
    'large_compact': (
        cluster_summary.query('n_cases >= @size_q90')
        .sort_values('median_dist_centroid').head(TOP_N)
    ),
    'large_dispersed': (
        cluster_summary.query('n_cases >= @size_q90')
        .sort_values('median_dist_centroid', ascending=False).head(TOP_N)
    ),
    'age_homogeneous': (
        cluster_summary.query('n_cases >= @MIN_CLUSTER_SIZE')
        .sort_values(['age_entropy', 'n_cases'], ascending=[True, False]).head(TOP_N)
    ),
    'age_mixed': (
        cluster_summary.query('n_cases >= @MIN_CLUSTER_SIZE')
        .sort_values(['age_entropy', 'n_cases'], ascending=[False, False]).head(TOP_N)
    ),
    'sex_skewed': (
        cluster_summary.query('n_cases >= @MIN_CLUSTER_SIZE')
        .sort_values(['dominant_sex_prop', 'n_cases'], ascending=[False, False]).head(TOP_N)
    ),
    'most_deprived': (
        cluster_summary.query('n_cases >= @MIN_CLUSTER_SIZE')
        .sort_values(['mean_simd', 'n_cases'], ascending=[True, False]).head(TOP_N)
    ),
    'least_deprived': (
        cluster_summary.query('n_cases >= @MIN_CLUSTER_SIZE')
        .sort_values(['mean_simd', 'n_cases'], ascending=[False, False]).head(TOP_N)
    ),
    'simd_mixed': (
        cluster_summary.query('n_cases >= @MIN_CLUSTER_SIZE')
        .sort_values(['simd_entropy', 'n_cases'], ascending=[False, False]).head(TOP_N)
    ),
    'multi_board': (
        cluster_summary.query('n_health_boards > 1')
        .sort_values(['n_health_boards', 'n_cases'], ascending=[False, False]).head(TOP_N)
    ),
    'lineage_mixed': (
        cluster_summary.query('n_cases >= @MIN_CLUSTER_SIZE')
        .sort_values(['lineage_entropy', 'n_cases'], ascending=[False, False]).head(TOP_N)
    ),
}, names=['selection_reason']).reset_index(level=0).reset_index(drop=True)

shortlist = shortlist.drop_duplicates(subset='cluster_id')

display_cols = [
    'selection_reason', 'cluster_id', 'n_cases',
    'median_dist_centroid', 'dominant_age_band', 'dominant_age_band_prop',
    'dominant_sex', 'dominant_sex_prop',
    'mean_simd', 'dominant_simd_quintile',
    'dominant_health_board', 'n_health_boards',
    'dominant_lineage', 'n_lineages',
]
print(f'Rule-based shortlist: {len(shortlist)} unique clusters')
shortlist[display_cols]

---
## 11. KMeans Meta-Typology

In [ ]:
feature_cols = [
    'n_cases', 'median_dist_centroid', 'max_dist_centroid',
    'date_range_days',
    'sex_entropy', 'age_entropy', 'dominant_sex_prop', 'dominant_age_band_prop',
    'mean_simd', 'simd_entropy', 'prop_simd_q1',
    'hb_entropy', 'lineage_entropy', 'dominant_lineage_prop',
]

features = cluster_summary[feature_cols].dropna().copy()
for col in ['n_cases', 'median_dist_centroid', 'max_dist_centroid']:
    features[col] = np.log1p(features[col])

X = StandardScaler().fit_transform(features)
# Note: KMeans index now corresponds only to rows with no NaN in feature_cols

inertias = []
K_range  = range(2, 13)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init='auto')
    inertias.append(km.fit(X).inertia_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(K_range), inertias, marker='o', color='steelblue')
ax.set_xlabel('k'); ax.set_ylabel('Inertia')
ax.set_title('Elbow plot — choose k at the bend')
plt.tight_layout(); plt.show()


In [ ]:
# ── Set k after inspecting the elbow plot ────────────────────────────────────
K_META = 5
# ─────────────────────────────────────────────────────────────────────────────

kmeans = KMeans(n_clusters=K_META, random_state=RANDOM_SEED, n_init='auto')
cluster_summary = cluster_summary.reset_index(drop=True)
cluster_summary['meta_cluster'] = kmeans.fit_predict(X)

meta_summary = (
    cluster_summary.groupby('meta_cluster').agg(
        n_molecular_clusters  =('cluster_id',           'count'),
        median_cases          =('n_cases',               'median'),
        median_spatial_spread =('median_dist_centroid',  'median'),
        median_age_entropy    =('age_entropy',           'median'),
        median_mean_simd      =('mean_simd',             'median'),
        median_simd_entropy   =('simd_entropy',          'median'),
        median_lin_entropy    =('lineage_entropy',       'median'),
        pct_multi_board       =('is_multi_board',        'mean'),
    ).reset_index()
)
print(f'Meta-cluster summary (k={K_META}):')
meta_summary

In [ ]:
colors = cm.tab10(np.linspace(0, 1, K_META))
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for mc in range(K_META):
    mask = cluster_summary['meta_cluster'] == mc
    kw   = dict(s=22, alpha=0.6, color=colors[mc], label=f'MC {mc}')
    axes[0].scatter(cluster_summary.loc[mask, 'n_cases'],
                    cluster_summary.loc[mask, 'median_dist_centroid'], **kw)
    axes[1].scatter(cluster_summary.loc[mask, 'age_entropy'],
                    cluster_summary.loc[mask, 'mean_simd'], **kw)
    axes[2].scatter(cluster_summary.loc[mask, 'simd_entropy'],
                    cluster_summary.loc[mask, 'lineage_entropy'], **kw)

axes[0].set_xlabel('Size'); axes[0].set_ylabel('Median dist centroid (m)')
axes[0].set_title('Size vs spatial spread'); axes[0].legend(fontsize=8, ncol=2)
axes[1].set_xlabel('Age entropy'); axes[1].set_ylabel('Mean SIMD quintile')
axes[1].set_title('Age diversity vs deprivation')
axes[2].set_xlabel('SIMD entropy'); axes[2].set_ylabel('Lineage entropy')
axes[2].set_title('Deprivation diversity vs lineage diversity')

plt.suptitle(f'Meta-cluster assignments (k={K_META})', fontsize=13)
plt.tight_layout(); plt.show()

### 11a. Representatives and extremes per meta-cluster

In [ ]:
representatives = []
for m in sorted(cluster_summary['meta_cluster'].unique()):
    idx    = cluster_summary.index[cluster_summary['meta_cluster'] == m]
    X_sub  = X[idx]
    centre = X_sub.mean(axis=0).reshape(1, -1)
    pos, _ = pairwise_distances_argmin_min(centre, X_sub)
    representatives.append(cluster_summary.loc[idx[pos[0]]].copy())

representatives_df = pd.DataFrame(representatives)
representatives_df['selection_reason'] = 'meta_representative'

extreme_examples = []
for m, g in cluster_summary.groupby('meta_cluster'):
    extreme_examples.append(g.sort_values('n_cases', ascending=False).head(1))
    extreme_examples.append(g.sort_values('median_dist_centroid', ascending=False).head(1))
    extreme_examples.append(g.sort_values('age_entropy', ascending=False).head(1))
    extreme_examples.append(g.sort_values('mean_simd').head(1))              # most deprived
    extreme_examples.append(g.sort_values('simd_entropy', ascending=False).head(1))
    extreme_examples.append(g.sort_values('lineage_entropy', ascending=False).head(1))

extreme_df = (
    pd.concat(extreme_examples).drop_duplicates(subset='cluster_id').reset_index(drop=True)
)
extreme_df['selection_reason'] = 'meta_extreme'

final_selection = (
    pd.concat([shortlist, representatives_df, extreme_df])
    .drop_duplicates(subset='cluster_id')
    .reset_index(drop=True)
)

print(f'Representatives: {len(representatives_df)}')
print(f'Extremes:        {len(extreme_df)}')
print(f'Final selection: {len(final_selection)} unique clusters')
final_selection[display_cols].head(20)

---
## 12. Cluster Dissection — Case-Level Plots

In [ ]:
def compare_cluster_to_overall(df_full, cluster_id, variable):
    g = df_full[df_full['cluster_id'] == cluster_id]
    cluster_dist = observed_value_counts(g[variable], normalize=True)
    overall_dist = observed_value_counts(df_full[variable], normalize=True)
    comp = pd.DataFrame({'cluster': cluster_dist, 'overall': overall_dist}).fillna(0)
    comp['difference'] = comp['cluster'] - comp['overall']
    if hasattr(df_full[variable], 'cat') and df_full[variable].cat.ordered:
        comp = comp.sort_index()
    else:
        comp = comp.sort_values('difference', ascending=False)
    return comp


def plot_cluster_detail(df_full, cluster_summary_df, cluster_id, figsize=(18, 17)):
    """
    11-panel dissection figure (4 rows × 3 cols; metadata spans rows 3–4 col 2):
    Row 1: spatial map | distance distribution | age band vs overall
    Row 2: sex | SIMD vs overall | health board
    Row 3: lineage | SIMD enrichment | metadata text (spans rows 3–4)
    Row 4: case timeline | cumulative case curve | (metadata continued)
    """
    g    = df_full[df_full['cluster_id'] == cluster_id].copy()
    meta = cluster_summary_df[cluster_summary_df['cluster_id'] == cluster_id].iloc[0]

    cx, cy = meta['centroid_x'], meta['centroid_y']
    dist   = np.sqrt((g['dz_xcoord'] - cx)**2 + (g['dz_ycoord'] - cy)**2)

    age_comp  = compare_cluster_to_overall(df_full, cluster_id, 'age_band')
    sex_comp  = compare_cluster_to_overall(df_full, cluster_id, 'sex')
    simd_comp = compare_cluster_to_overall(df_full, cluster_id, 'dz_simd_quintile')

    dates_sorted = g['collection_date'].dropna().sort_values()

    fig = plt.figure(figsize=figsize)
    gs  = GridSpec(4, 3, figure=fig, hspace=0.60, wspace=0.42)
    bw  = 0.38

    # ── Row 1 ──────────────────────────────────────────────────────────────────
    # 1. Spatial map coloured by SIMD
    ax1 = fig.add_subplot(gs[0, 0])
    sc  = ax1.scatter(g['dz_xcoord'], g['dz_ycoord'],
                      c=g['dz_simd_quintile'].astype(int),
                      cmap='RdYlGn', alpha=0.75, s=35, vmin=1, vmax=5)
    ax1.scatter([cx], [cy], marker='+', s=200, color='black', zorder=5, label='centroid')
    plt.colorbar(sc, ax=ax1, label='SIMD quintile')
    ax1.set_xlabel('X (m BNG)'); ax1.set_ylabel('Y (m BNG)')
    ax1.set_title(f'Spatial map (n={len(g)})\nColour=SIMD quintile')
    ax1.set_aspect('equal')

    # 2. Distance to centroid
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.hist(dist, bins=15, edgecolor='white', color='steelblue')
    ax2.axvline(dist.median(), color='red', linestyle='--',
                label=f'Median={dist.median():,.0f} m')
    ax2.set_xlabel('Distance to centroid (m)'); ax2.set_ylabel('Cases')
    ax2.set_title('Spatial spread'); ax2.legend(fontsize=8)

    # 3. Age band vs overall
    ax3 = fig.add_subplot(gs[0, 2])
    x3  = np.arange(len(age_comp))
    ax3.bar(x3 - bw/2, age_comp['cluster'], width=bw,
            label='Cluster', color='steelblue', alpha=0.85)
    ax3.bar(x3 + bw/2, age_comp['overall'],  width=bw,
            label='Overall', color='grey', alpha=0.6)
    ax3.set_xticks(x3)
    ax3.set_xticklabels(age_comp.index, rotation=45, ha='right', fontsize=8)
    ax3.set_ylabel('Proportion'); ax3.set_title('Age band'); ax3.legend(fontsize=8)

    # ── Row 2 ──────────────────────────────────────────────────────────────────
    # 4. Sex vs overall
    ax4 = fig.add_subplot(gs[1, 0])
    x4  = np.arange(len(sex_comp))
    ax4.bar(x4 - bw/2, sex_comp['cluster'], width=bw,
            label='Cluster', color='steelblue', alpha=0.85)
    ax4.bar(x4 + bw/2, sex_comp['overall'],  width=bw,
            label='Overall', color='grey', alpha=0.6)
    ax4.set_xticks(x4); ax4.set_xticklabels(sex_comp.index, fontsize=9)
    ax4.set_ylabel('Proportion'); ax4.set_title('Sex'); ax4.legend(fontsize=8)

    # 5. SIMD vs overall
    ax5 = fig.add_subplot(gs[1, 1])
    x5  = np.arange(len(simd_comp))
    ax5.bar(x5 - bw/2, simd_comp['cluster'], width=bw,
            label='Cluster', color='steelblue', alpha=0.85)
    ax5.bar(x5 + bw/2, simd_comp['overall'],  width=bw,
            label='Overall', color='grey', alpha=0.6)
    ax5.set_xticks(x5)
    ax5.set_xticklabels([f'Q{int(q)}' for q in simd_comp.index], fontsize=9)
    ax5.set_ylabel('Proportion')
    ax5.set_title('SIMD quintile\n(Q1=most deprived)'); ax5.legend(fontsize=8)

    # 6. Health board breakdown
    ax6 = fig.add_subplot(gs[1, 2])
    hb_counts = observed_value_counts(g['health_board'])
    hb_counts.plot(kind='barh', ax=ax6, color='steelblue', edgecolor='white')
    ax6.set_xlabel('Cases'); ax6.set_title('Health board breakdown')
    ax6.tick_params(axis='y', labelsize=7)

    # ── Row 3 ──────────────────────────────────────────────────────────────────
    # 7. Lineage composition
    ax7 = fig.add_subplot(gs[2, 0])
    lin_counts = observed_value_counts(g['pango_lineage'])
    lin_counts.plot(kind='bar', ax=ax7, color='steelblue', edgecolor='white')
    ax7.set_ylabel('Cases'); ax7.set_title('Pango lineage')

    # 8. SIMD enrichment (cluster minus overall)
    ax8 = fig.add_subplot(gs[2, 1])
    diff = simd_comp['difference']
    bcol = ['#d73027' if d > 0 else '#4575b4' for d in diff]
    ax8.bar(np.arange(len(diff)), diff, color=bcol, edgecolor='white')
    ax8.axhline(0, color='black', linewidth=0.8)
    ax8.set_xticks(np.arange(len(diff)))
    ax8.set_xticklabels([f'Q{int(q)}' for q in simd_comp.index])
    ax8.set_ylabel('Cluster - Overall proportion')
    ax8.set_title('SIMD enrichment vs overall')
    ax8.legend(handles=[
        mpatches.Patch(color='#d73027', label='Enriched'),
        mpatches.Patch(color='#4575b4', label='Depleted'),
    ], fontsize=8)

    # ── Row 4 ──────────────────────────────────────────────────────────────────
    # 9. Case timeline
    ax9 = fig.add_subplot(gs[3, 0])
    if not dates_sorted.empty:
        date_counts = dates_sorted.dt.floor('D').value_counts().sort_index()
        ax9.bar(date_counts.index, date_counts.values,
                color='steelblue', edgecolor='white', width=0.8)
        med_d = meta.get('median_collection_date')
        if pd.notna(med_d):
            ax9.axvline(pd.Timestamp(med_d), color='red', linestyle='--', linewidth=1.2,
                        label=f"Median: {pd.Timestamp(med_d).date()}")
            ax9.legend(fontsize=7)
    dur = int(meta['date_range_days']) if pd.notna(meta.get('date_range_days')) else 'N/A'
    ax9.set_xlabel('Collection date'); ax9.set_ylabel('Cases')
    ax9.set_title(f'Case timeline ({dur} day span)')
    ax9.tick_params(axis='x', rotation=45, labelsize=7)

    # 10. Cumulative case curve
    ax10 = fig.add_subplot(gs[3, 1])
    if not dates_sorted.empty:
        ax10.step(dates_sorted.values, np.arange(1, len(dates_sorted) + 1),
                  where='post', color='steelblue', linewidth=1.5)
        ax10.set_ylim(0)
    ax10.set_xlabel('Collection date'); ax10.set_ylabel('Cumulative cases')
    ax10.set_title('Cumulative case curve')
    ax10.tick_params(axis='x', rotation=45, labelsize=7)

    # 11. Metadata (spans rows 3–4, col 2)
    ax11 = fig.add_subplot(gs[2:4, 2])
    ax11.axis('off')
    mc_label = str(int(meta['meta_cluster'])) if 'meta_cluster' in meta.index else 'N/A'
    def _fmt_date(key):
        v = meta.get(key)
        return str(pd.Timestamp(v).date()) if pd.notna(v) else 'N/A'
    dur_d = int(meta['date_range_days']) if pd.notna(meta.get('date_range_days')) else 'N/A'
    info = (
        f"Cluster ID:          {cluster_id}\n"
        f"Cases (n):           {int(meta['n_cases'])}\n"
        f"Datazones (unique):  {int(meta['n_datazones'])}\n"
        f"Meta-cluster:        {mc_label}\n\n"
        f"── Temporal ──────────────────────\n"
        f"Window midpoint:     {_fmt_date('wn_mid_date')}\n"
        f"First case:          {_fmt_date('min_collection_date')}\n"
        f"Last case:           {_fmt_date('max_collection_date')}\n"
        f"Median date:         {_fmt_date('median_collection_date')}\n"
        f"Date span (days):    {dur_d}\n\n"
        f"── Spatial ───────────────────────\n"
        f"Median dist (m):     {meta['median_dist_centroid']:,.0f}\n"
        f"Max dist (m):        {meta['max_dist_centroid']:,.0f}\n\n"
        f"── Demographics ──────────────────\n"
        f"Dominant sex:        {meta['dominant_sex']} ({meta['dominant_sex_prop']:.1%})\n"
        f"Dominant age band:   {meta['dominant_age_band']} "
        f"({meta['dominant_age_band_prop']:.1%})\n"
        f"Age entropy:         {meta['age_entropy']:.3f}\n\n"
        f"── Deprivation ───────────────────\n"
        f"Mean SIMD:           {meta['mean_simd']:.2f}\n"
        f"Dominant SIMD Q:     Q{int(meta['dominant_simd_quintile'])} "
        f"({meta['dominant_simd_prop']:.1%})\n"
        f"SIMD entropy:        {meta['simd_entropy']:.3f}\n"
        f"Prop Q1 (deprived):  {meta['prop_simd_q1']:.1%}\n\n"
        f"── Geography ─────────────────────\n"
        f"Health boards (n):   {int(meta['n_health_boards'])}\n"
        f"Dominant HB:         {meta['dominant_health_board']}\n\n"
        f"── Lineage ───────────────────────\n"
        f"Dominant lineage:    {meta['dominant_lineage']} "
        f"({meta['dominant_lineage_prop']:.1%})\n"
        f"Lineages (n):        {int(meta['n_lineages'])}"
    )
    ax11.text(0.03, 0.97, info, transform=ax11.transAxes,
              fontsize=8.5, verticalalignment='top', fontfamily='monospace',
              bbox=dict(boxstyle='round', facecolor='#f5f5f5', alpha=0.85))
    ax11.set_title('Cluster metadata')

    plt.suptitle(f'Cluster {cluster_id} — detailed dissection', fontsize=14, y=1.01)
    plt.show()


print('Plotting functions defined.')


---
## 13. Dissect Selected Clusters

In [ ]:
# ── Choose clusters to dissect ────────────────────────────────────────────────
ids_to_dissect = final_selection['cluster_id'].head(6).tolist()

# Or specify manually:
# ids_to_dissect = ['CL0001', 'CL0042', 'CL0123']
# ─────────────────────────────────────────────────────────────────────────────

print(f'Dissecting {len(ids_to_dissect)} clusters: {ids_to_dissect}')

In [ ]:
for cid in ids_to_dissect:
    plot_cluster_detail(df, cluster_summary, cid)

---
## 14. Enrichment Heatmaps

In [ ]:
def enrichment_heatmap(df_full, cluster_ids, variable, title,
                        figsize=None, xticklabel_fn=None):
    """Heatmap of (cluster proportion - overall proportion) for `variable`."""
    rows = []
    for cid in cluster_ids:
        comp = compare_cluster_to_overall(df_full, cid, variable)
        for cat, row in comp.iterrows():
            rows.append({'cluster_id': cid, variable: cat, 'difference': row['difference']})

    pivot = (
        pd.DataFrame(rows)
        .pivot(index='cluster_id', columns=variable, values='difference')
        .fillna(0)
    )

    if figsize is None:
        figsize = (max(8, len(pivot.columns) * 0.9 + 2),
                   max(4, len(pivot) * 0.35 + 1.5))

    fig, ax = plt.subplots(figsize=figsize)
    vmax = pivot.abs().max().max()
    im   = ax.imshow(pivot.values, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
    plt.colorbar(im, ax=ax, label='Cluster proportion - Overall proportion')
    xlabels = pivot.columns if xticklabel_fn is None else [xticklabel_fn(c) for c in pivot.columns]
    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels(xlabels, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=7)
    ax.set_title(f'{title}\n(red=enriched vs overall, blue=depleted)')
    plt.tight_layout(); plt.show()
    return pivot


sel_ids = final_selection['cluster_id'].tolist()

print('Age band enrichment:')
age_pivot = enrichment_heatmap(df, sel_ids, 'age_band', 'Age band enrichment')

In [ ]:
print('SIMD quintile enrichment:')
simd_pivot = enrichment_heatmap(
    df, sel_ids, 'dz_simd_quintile', 'SIMD quintile enrichment',
    xticklabel_fn=lambda q: f'Q{int(q)}'
)

In [ ]:
print('Health board enrichment:')
hb_pivot = enrichment_heatmap(
    df, sel_ids, 'health_board', 'Health board enrichment',
    figsize=(16, max(4, len(sel_ids) * 0.35 + 1.5))
)

---
## 15. Catalogue Summary Statistics

In [ ]:
cs = cluster_summary
print('=' * 60)
print('FULL CLUSTER CATALOGUE SUMMARY')
print('=' * 60)
print(f'Total clusters (size >= {MIN_CLUSTER_SIZE}): {len(cs):,}')
print(f'Total sequences in those clusters:  {int(cs["n_cases"].sum()):,}')

print('\nCluster size:')
print(f'  Median {cs["n_cases"].median():.0f} | Mean {cs["n_cases"].mean():.1f} | Max {cs["n_cases"].max():.0f}')

print('\nDate range (days, intra-window):')
print(f'  Median {cs["date_range_days"].median():.0f} | '
      f'IQR {cs["date_range_days"].quantile(0.25):.0f}–'
      f'{cs["date_range_days"].quantile(0.75):.0f} | '
      f'Max {cs["date_range_days"].max():.0f}')
print(f'  Proportion spanning >=7 days: {(cs["date_range_days"] >= 7).mean():.1%}')

print('\nMedian distance to centroid (m):')
print(f'  Median {cs["median_dist_centroid"].median():,.0f}')
print(f'  IQR    {cs["median_dist_centroid"].quantile(0.25):,.0f} - '
      f'{cs["median_dist_centroid"].quantile(0.75):,.0f}')

print('\nMean SIMD quintile (1=most deprived, 5=least):')
print(f'  Median {cs["mean_simd"].median():.2f} | '
      f'IQR {cs["mean_simd"].quantile(0.25):.2f}-{cs["mean_simd"].quantile(0.75):.2f}')

print('\nAge entropy:')
print(f'  Median {cs["age_entropy"].median():.3f} | '
      f'IQR {cs["age_entropy"].quantile(0.25):.3f}-{cs["age_entropy"].quantile(0.75):.3f}')

print('\nSIMD entropy:')
print(f'  Median {cs["simd_entropy"].median():.3f} | '
      f'IQR {cs["simd_entropy"].quantile(0.25):.3f}-{cs["simd_entropy"].quantile(0.75):.3f}')

print('\nLineage entropy:')
print(f'  Median {cs["lineage_entropy"].median():.3f} | '
      f'IQR {cs["lineage_entropy"].quantile(0.25):.3f}-{cs["lineage_entropy"].quantile(0.75):.3f}')

print(f'\nMulti-board clusters: {cs["is_multi_board"].sum()} ({cs["is_multi_board"].mean():.1%})')
print(f'Mixed-lineage clusters: {cs["is_mixed_lineage"].sum()} ({cs["is_mixed_lineage"].mean():.1%})')

for label, col in [
    ('Size',        'size_type'),
    ('Temporal',    'temporal_type'),
    ('Spatial',     'spatial_type'),
    ('Age',         'age_type'),
    ('Deprivation', 'deprivation_type'),
    ('Geographic',  'geographic_type'),
    ('Lineage',     'lineage_type'),
    ('Meta-cluster','meta_cluster'),
]:
    print(f'\n{label} breakdown:')
    print(cs[col].value_counts().sort_index().to_string())


---
## 16. Export

In [ ]:
cluster_summary.to_csv('cluster_summary.csv', index=False)
final_selection.to_csv('selected_clusters.csv', index=False)
age_pivot.to_csv('age_band_enrichment.csv')
simd_pivot.to_csv('simd_enrichment.csv')
hb_pivot.to_csv('health_board_enrichment.csv')

print('Uncomment lines above to export.')
print(f'cluster_summary:  {cluster_summary.shape}')
print(f'final_selection:  {final_selection.shape}')

---
## Notes and Caveats

1. **Residential datazone != infection location.** Spatial summaries reflect where cases live.
2. **Molecular clusters != confirmed transmission chains.** They reflect genetic similarity.
3. **Large clusters may reflect sampling intensity** as much as true transmission.
4. **SIMD quintiles are area-level**, not individual-level, deprivation measures.
5. **Multi-board clusters** may reflect patient movement or cross-board sequencing programmes rather than cross-board transmission.
6. **Mixed-lineage clusters** may indicate co-circulating strains, mis-assigned sequences, or cluster ID re-use.
7. **`collection_date` gives intra-window timing** — the full span of a cluster is bounded by the three-week window width. Cross-window temporal trends can be approximated by aggregating cluster summaries against `wn_mid_date`, but overlapping windows share sequences so consecutive windows are not independent.
8. **Age, sex, SIMD, and lineage patterns may reflect testing or sequencing bias** in addition to true epidemiological signals.

**Preferred phrasing:**
> *"This cluster is spatially compact, concentrated in SIMD quintile 1 datazones of NHS Greater Glasgow and Clyde, and predominantly comprises adults aged 20-39. This is consistent with localised transmission in a deprived urban community, though residential location and sampling patterns may also contribute."*

**Avoid:**
> *"This cluster proves transmission among young adults in deprived areas of Glasgow."*